# 📝 토픽 모델링 심화 과제 LV1(기초) — 임베딩·유사도·BERTopic 기초

> 이 단원에서 배운 **문서 임베딩**(문장→768차원 벡터)·**코사인 유사도**·**UMAP 시각화**·**BERTopic**(주제 추출)을 **선크림 리뷰** 데이터로 **한 문제에 하나씩** 확인하는 과제입니다.

## 풀이 방법
1. 맨 위 **제공 코드 셀**(라이브러리·토크나이저·임베딩 모델)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
3. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
4. 막히면 `힌트` 를 펼쳐 보세요.

> **자가채점이 없는 문제**: 그래프 문제(6·11)와 서술형(1)입니다. 정답 노트북의 완성 그래프·모범 서술과 비교하세요.

- 데이터는 `data/reviews_sun.csv`(선크림 리뷰 545개, 열: `rating` 별점 1~5, `text` 리뷰 본문) 를 씁니다.
- 리뷰 본문을 미리 임베딩한 벡터가 `data/reviews_sun_embeddings.npy`(545×768) 에, 2D 시각화용 재현 좌표가 `data/reviews_sun_umap2d.npy`(545×2) 에 저장돼 있습니다. 임베딩은 만들지 말고 `np.load(...)` 로 불러오세요(모두 같은 결과가 나오도록).
- 문제마다 필요한 파일을 새로 불러오면 앞 문제의 변형에 영향받지 않아요.

화이팅!

아래 셀을 먼저 실행해 이 단원 라이브러리와 한국어 토크나이저를 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 토크나이저를 준비합니다.
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from kiwipiepy import Kiwi

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

# 지난 단원에서 배운 형태소 분석 — c-TF-IDF 키워드를 한국어 명사로 뽑는 토크나이저(자바가 필요없는 kiwipiepy)
kiwi = Kiwi()

# 불용어 — 11일차에서 배운 방식 그대로: 공개 일반 목록 + 이 데이터의 도메인 불용어
with open('data/stopwords_ko.json', encoding='utf-8') as f:
    STOPWORDS_GENERAL = set(json.load(f))     # 11일차에서 받아 둔 공개 목록 679개

# 이 데이터에서만 무의미한 고빈도어 — 빈도표를 보고 사람이 고른다(11일차 4절)
STOPWORDS_DOMAIN = {'제품', '구매', '사용', '정말', '진짜', '완전', '그냥', '너무', '정도', '많이'}

# 반대로 일반 목록이 '여기서는 의미 있는 말'까지 지우기도 한다 — 되살릴 단어
# ('아이'·'시간' 은 일반 불용어지만, 이 리뷰에서는 '아이에게 사 준 밴드'처럼 주제를 가른다)
KEEP_WORDS = {'아이', '시간'}
KOREAN_STOPWORDS = (STOPWORDS_GENERAL | STOPWORDS_DOMAIN) - KEEP_WORDS

def korean_tokenizer(text):
    """문서에서 의미있는 명사(2글자 이상)만 골라 돌려줍니다."""
    return [t.form for t in kiwi.tokenize(str(text))
            if t.tag.startswith('NN') and len(t.form) > 1 and t.form not in KOREAN_STOPWORDS]

이어서 지난 단원에서 배운 **한국어 임베딩 모델**을 불러옵니다(문장→768차원 벡터). 리뷰 본문은 이미 이 모델로 임베딩해 `.npy` 로 저장해 두었습니다.

In [ ]:
# [제공 코드] 지난 단원에서 배운 한국어 임베딩 모델을 불러옵니다(문장→768차원 벡터).
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# 임베딩·2D 좌표 — 저장된 파일이 있으면 그대로 쓰고, 없으면 지금 만들어 저장합니다
emb_path = 'data/reviews_sun_embeddings.npy'
umap_path = 'data/reviews_sun_umap2d.npy'
if not os.path.exists(emb_path):
    print('저장된 임베딩이 없어 지금 만듭니다 — 수 분 걸릴 수 있어요')
    texts = pd.read_csv('data/reviews_sun.csv')['text'].astype(str).tolist()
    np.save(emb_path, emb_model.encode(texts, show_progress_bar=False))
if not os.path.exists(umap_path):
    print('저장된 2차원 좌표가 없어 지금 만듭니다')
    np.save(umap_path, UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                            metric='cosine', random_state=42).fit_transform(np.load(emb_path)))

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 별점 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·요약
reviews = pd.read_csv('data/reviews_sun.csv')
print("행·열 크기:", reviews.shape)
print("\n[앞 5행] head()"); display(reviews.head())
print("\n[열·자료형·결측] info()"); reviews.info()
print("\n[별점 요약] describe()"); display(reviews.describe())

## 1. 데이터 살펴보기 (서술형)
**배경**: 분석에 들어가기 전에 데이터를 **눈으로 파악**하는 것이 첫걸음입니다. 위 `데이터 살펴보기` 셀의 출력을 보고, 이 선크림 리뷰 데이터에 대해 알게 된 사실을 정리해 보세요.

**요구사항**:
- 아래 서술 셀에 이 데이터에 대한 **관찰 2~3가지**를 문장으로 적으세요.
- 예를 들어: 리뷰가 몇 개인지, 별점(`rating`)의 분포·평균, 결측치가 있는지, 리뷰 본문(`text`)이 어떤 내용인지 등 눈에 띄는 점을 적으면 됩니다.
- 정답은 하나가 아닙니다. 출력에서 실제로 확인되는 사실이면 됩니다.

> 이 문제는 자가채점(assert)이 없습니다. 아래 서술 셀에 직접 문장을 적고, 정답 노트북의 모범 서술과 비교해 보세요.

*(여기에 관찰을 서술하세요)*

## 2. 임베딩 불러오기와 모양 확인
**배경**: 리뷰 본문 545개는 이미 지난 단원의 임베딩 모델로 **768차원 벡터**로 바꿔 `.npy` 파일에 저장돼 있습니다. 분석의 출발점은 이 임베딩 행렬을 불러와 **모양(shape)** 을 확인하는 것입니다.

**요구사항**:
- `np.load('data/reviews_sun_embeddings.npy')` 로 임베딩을 불러와 변수 `emb` 에 담으세요.
- `emb.shape` 가 `(545, 768)` 인지 확인하세요(리뷰 545개 × 각 768차원).

**예시**
```
emb.shape  →  (545, 768)
```
<details><summary>힌트</summary>

```text
접근방법:
- 미리 저장된 임베딩 파일을 numpy 로 불러온 뒤 shape 속성을 확인한다.

세부구현:
1. np.load 에 임베딩 파일 경로를 넘겨 emb 에 담는다
2. emb.shape 를 출력해 (리뷰 수, 차원) 을 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert emb.shape == (545, 768)
print("✅ 문제2 통과!")

## 3. 리뷰 한 개의 벡터 살펴보기
**배경**: 임베딩 행렬에서 **한 리뷰의 벡터**는 한 행입니다. 첫 번째 리뷰(0번)의 벡터를 꺼내 차원과 실제 숫자 몇 개를 확인해, 리뷰 하나가 768개의 숫자로 표현된다는 것을 눈으로 봅시다.

**요구사항**:
- 문제 2 에서 불러온 `emb` 에서 **0번 리뷰의 벡터**를 꺼내 변수 `vec` 에 담으세요(`emb[0]`).
- `vec.shape` 가 `(768,)` 인지 확인하세요(리뷰 한 개는 768차원 벡터 하나).
- `vec[:5]` 로 앞쪽 다섯 개 값을 출력해, 벡터가 실수들로 채워져 있음을 확인하세요.

**예시**
```
vec.shape   →  (768,)
vec[:5]     →  앞쪽 다섯 개의 실수 값(예: [ 0.31 -0.12 ... ])
```
<details><summary>힌트</summary>

```text
접근방법:
- 2차원 임베딩에서 한 행을 인덱싱하면 1차원 벡터가 된다.

세부구현:
1. emb 의 0번 행을 vec 에 담는다
2. vec 의 shape 를 확인하고 앞쪽 다섯 개 값을 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert vec.shape == (768,)
print("✅ 문제3 통과!")

## 4. 코사인 유사도 — 0번 리뷰와 가장 비슷한 리뷰 top3
**배경**: 두 문서가 얼마나 비슷한지는 **코사인 유사도**(벡터가 이루는 각도)로 잽니다. 0번 리뷰와 나머지 모든 리뷰의 유사도를 구해, **자기 자신을 빼고** 가장 비슷한 3개를 찾아봅시다.

**요구사항**:
- `cosine_similarity` 로 **0번 리뷰**와 모든 리뷰의 유사도를 구해 변수 `sims` 에 담으세요(길이 545의 1차원 배열이 되도록).
- `sims` 가 **큰 순서**로 리뷰 인덱스를 정렬한 뒤, **자기 자신(0번)을 빼고** 상위 3개 인덱스를 변수 `top3` 에 담으세요. (자기 자신과의 유사도는 항상 1.0 이라 늘 맨 앞에 옵니다.)

**예시**
```
len(sims)     →  545
len(top3)     →  3
0 in top3     →  False   (자기 자신은 제외)
```
<details><summary>힌트</summary>

```text
접근방법:
- 0번 벡터와 전체 임베딩의 코사인 유사도를 한 번에 구한다.
- 유사도가 큰 순서로 인덱스를 정렬하고, 맨 앞(자기 자신)을 건너뛴 다음 3개를 고른다.

세부구현:
1. cosine_similarity 에 (emb 의 0번 한 행, 전체 emb) 을 넘겨 첫 행을 sims 에 담는다
2. argsort 를 뒤집어 유사도 내림차순 인덱스를 만든다
3. 그 인덱스의 1번~3번(0번=자기자신 제외)을 top3 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(sims) == 545
assert len(top3) == 3
assert 0 not in list(top3)
# 값을 지어내지 않았는지 — 채점 셀이 emb 로 직접 다시 계산해 대조한다
_s0 = cosine_similarity(emb[0:1], emb)[0]
# (순서는 따지지 않는다 — argpartition 처럼 순서를 보장하지 않는 방법도 정답으로 인정)
assert set(top3) == set(np.argsort(_s0)[::-1][1:4]), '유사도를 실제로 계산해 고른 top3 가 아닙니다'
print("✅ 문제4 통과!")

## 5. HDBSCAN 으로 직접 군집화
**배경**: 지금까지는 리뷰 **둘 사이**의 유사도만 봤습니다. 이번엔 **545개 전체를 한꺼번에 군집으로** 묶어 봅니다. 교안에서 배운 **HDBSCAN** 은 군집를 몇 개 만들지 사람이 정해 주지 않아도 **빽빽하게 모인 곳만** 군집으로 잡고, 어디에도 안 붙는 리뷰는 **−1(노이즈)** 로 남깁니다.

**요구사항**:
- 미리 계산된 2D 좌표 `data/reviews_sun_umap2d.npy`(545×2)를 불러와 변수 `coords` 에 담으세요.
- 교안과 같은 설정 `HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom')` 으로 `coords` 를 군집화해, 각 리뷰의 **군집 번호 배열**을 변수 `labels` 에 담으세요.
- 노이즈(−1)를 **뺀** 군집 개수를 변수 `n_groups` 에, 노이즈 리뷰 수를 변수 `n_noise_g` 에 각각 **정수**로 담아 출력하세요.

**예시**
```
len(labels)   →  545          (리뷰마다 군집 번호 하나)
n_groups      →  7 안팎의 정수 (노이즈 -1 은 군집으로 세지 않는다)
n_noise_g     →  275 안팎      (전체 545개의 절반쯤 — 한 상품 리뷰라 경계가 흐릿하다)
```
<details><summary>힌트</summary>

```text
접근방법:
- 2D 좌표를 불러와 HDBSCAN 에 그대로 넣으면 군집 번호 배열이 나온다(학습과 배정이 한 번에).
- 군집 개수는 번호의 종류 수에서 노이즈(-1)를 빼면 된다.

세부구현:
1. np.load 로 2D 좌표를 coords 에 담는다
2. HDBSCAN 을 지시된 설정으로 만들어 좌표에 대해 fit_predict 를 부르고 결과를 labels 에 담는다
3. labels 의 서로 다른 값 중 -1 을 뺀 개수를 n_groups 에 담는다
4. labels 가 -1 인 개수를 n_noise_g 에 담고 둘 다 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import numpy as np
assert len(labels) == 545, 'labels 는 리뷰마다 군집 번호 하나씩, 길이 545 여야 합니다'
assert n_groups == len(set(labels) - {-1}), 'n_groups 는 노이즈(-1)를 뺀 군집 개수여야 합니다'
assert n_noise_g == int((np.array(labels) == -1).sum()), 'n_noise_g 는 labels 에서 실제로 센 값이어야 합니다'
assert 3 <= n_groups <= 12, '군집 수가 예상 범위를 벗어났습니다 — 지시된 설정 그대로 군집했는지 확인하세요'
assert 200 <= n_noise_g <= 340, '노이즈 수가 예상 범위를 벗어났습니다 — 지시된 설정 그대로 군집했는지 확인하세요'
print("✅ 문제5 통과! (군집", n_groups, "개 / 노이즈", n_noise_g, "건)")

## 6. 임베딩을 2D 지도로 — UMAP 산점도
**배경**: 768차원 벡터는 눈으로 볼 수 없습니다. **UMAP** 은 고차원 임베딩을 의미를 최대한 보존하며 2차원으로 줄여 줍니다. 미리 계산된 2D 좌표를 불러와 산점도로 그려, 비슷한 리뷰끼리 뭉치는 모습을 확인해 봅시다.

**요구사항**:
- `np.load('data/reviews_sun_umap2d.npy')` 로 2D 좌표(545×2)를 불러와 변수 `coords` 에 담으세요.
- `plt.figure(figsize=(7, 6))` 로 새 그림을 연 뒤, `plt.scatter(coords[:, 0], coords[:, 1], s=12, alpha=0.6)` 로 산점도를 그리세요.
- 제목·축 이름을 달고 `plt.show()` 로 보여 주세요.

**예시**: 아래 완성 그래프와 같은 모양(545개 점이 넓게 퍼진 2D 지도)이면 됩니다. 덩어리가 뚜렷이 갈리지 않아도 정상입니다 — 모두 **한 상품(선크림)의 리뷰**라 주제가 서로 이어져 있기 때문입니다. 이 점이 문제 10 에서 노이즈가 많이 나오는 이유가 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 미리 저장된 2D 좌표를 불러와 x·y 열로 산점도를 그린다.
- 그리기 전에 plt.figure 로 새 그림을 연다.

세부구현:
1. np.load 로 2D 좌표를 coords 에 담는다
2. plt.figure 로 새 그림을 연다
3. coords 의 0열(x)·1열(y)로 scatter 를 그리고 제목·축 이름을 단 뒤 plt.show 로 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv1_q6_umap.png" width="560">

In [ ]:
# 여기에 코드를 작성하세요

## 7. BERTopic — 리뷰에서 주제 자동 추출
**배경**: **BERTopic** 은 임베딩으로 문서를 군집화한 뒤 각 군집의 대표 키워드를 뽑아 **주제(토픽)** 를 자동으로 찾아 줍니다. 표준 구성으로 선크림 리뷰의 토픽을 뽑아 봅시다. 미리 만든 임베딩을 넘기면 빠릅니다.

**요구사항**:
- `data/reviews_sun.csv` 의 `text` 열을 리스트로 만들어 변수 `docs` 에, 임베딩을 `np.load` 로 불러와 변수 `emb` 에 담으세요.
- 아래 표준 구성으로 BERTopic 모델을 만들어 변수 `topic_model` 에 담으세요(제공 코드 셀의 `emb_model`·`korean_tokenizer` 를 사용합니다):
```
umap_model = UMAP(n_components=5, n_neighbors=15, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean',
                        prediction_data=True)
vectorizer_model = CountVectorizer(tokenizer=korean_tokenizer, max_df=0.85)
topic_model = BERTopic(embedding_model=emb_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                    language='multilingual', verbose=False)
```
- `topic_model.fit_transform(docs, embeddings=emb)` 로 학습하고, 반환값을 **`topics, probs`** 에 담으세요. `topics`(각 문서의 토픽 번호)는 **문제 10 에서도 씁니다.**
- `topic_model.get_topic_info()` 를 변수 `topic_info` 에 담고 `display(topic_info)` 로 확인하세요. 표의 `Topic` 열에서 **−1 은 어디에도 안 묶인 노이즈** 문서입니다.

**예시**
```
display(topic_info)  →  Topic / Count / Name ... 열을 가진 표
노이즈(-1)를 뺀 토픽 수는 대략 3~15개 사이
```
<details><summary>힌트</summary>

```text
접근방법:
- 문서 리스트와 미리 만든 임베딩을 준비하고, 표준 구성 그대로 BERTopic 을 만들어 학습시킨다.
- 학습 후 get_topic_info 로 토픽 목록·문서 수를 표로 확인한다.

세부구현:
1. csv 의 text 열을 리스트로 만들어 docs 에, 임베딩을 np.load 로 emb 에 담는다
2. 표준 구성(UMAP·HDBSCAN·CountVectorizer)으로 topic_model 을 만든다
3. fit_transform 에 docs 와 embeddings=emb 를 넘긴다
4. get_topic_info 를 topic_info 에 담아 표로 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
n_topics = int((topic_info['Topic'] != -1).sum())   # 노이즈(-1) 제외 토픽 수
assert 3 <= n_topics <= 15
assert 'Count' in topic_info.columns
print("✅ 문제7 통과! (토픽 수:", n_topics, ")")

## 8. 토픽의 대표 키워드 보기
**배경**: 각 토픽이 **무엇에 관한 주제인지**는 대표 키워드로 읽습니다. `get_topic(토픽번호)` 는 그 토픽의 (단어, 점수) 쌍을 점수가 높은 순으로 돌려줍니다. 0번 토픽의 키워드를 살펴봅시다.

**요구사항**:
- 문제 7 의 `topic_model` 을 그대로 사용하세요.
- `topic_model.get_topic(0)` 으로 0번 토픽의 키워드 목록을 변수 `keywords` 에 담으세요.
- `keywords` 는 `(단어, 점수)` 튜플의 리스트입니다. 앞쪽 몇 개를 출력해 어떤 주제인지 읽어 보세요.

**예시**
```
len(keywords) >= 5           →  True
keywords[0]                  →  ('크림', 0.1055) 처럼 (단어, 점수) 튜플
```
<details><summary>힌트</summary>

```text
접근방법:
- 학습된 토픽 모델에서 0번 토픽의 키워드 목록을 가져온다. 각 원소는 (단어, 점수) 튜플이다.

세부구현:
1. topic_model.get_topic 에 0 을 넘겨 keywords 에 담는다
2. 앞쪽 원소를 출력해 (단어, 점수) 형태와 주제를 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(keywords) >= 5
assert isinstance(keywords[0], tuple) and len(keywords[0]) == 2
assert isinstance(keywords[0][0], str)
# 지어낸 값이 아니라 실제 모델에서 가져왔는지 대조
assert [w for w, _ in keywords[:5]] == [w for w, _ in topic_model.get_topic(0)[:5]]
print("✅ 문제8 통과!")

## 9. 토픽의 대표 문서 보기
**배경**: 키워드만으로 감이 안 오면 그 토픽을 가장 잘 대표하는 **실제 리뷰**를 읽어 봅니다. `get_representative_docs(토픽번호)` 는 그 토픽의 대표 문서들을 돌려줍니다.

**요구사항**:
- 문제 7 의 `topic_model` 을 그대로 사용하세요.
- `topic_model.get_representative_docs(0)` 으로 0번 토픽의 대표 문서 목록을 변수 `rep_docs` 에 담으세요.
- 첫 번째 대표 문서의 앞부분을 출력해, 문제 8 의 키워드와 내용이 맞는지 눈으로 확인해 보세요.

**예시**
```
len(rep_docs) >= 1        →  True
rep_docs[0]               →  0번 토픽을 대표하는 리뷰 본문(문자열)
```
<details><summary>힌트</summary>

```text
접근방법:
- 학습된 토픽 모델에서 0번 토픽의 대표 문서 목록을 가져와 첫 문서를 읽어 본다.

세부구현:
1. topic_model.get_representative_docs 에 0 을 넘겨 rep_docs 에 담는다
2. rep_docs 의 첫 문서 앞부분을 출력해 키워드와 대조한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(rep_docs) >= 1
assert isinstance(rep_docs[0], str)
# 지어낸 문자열이 아니라 실제 리뷰 본문인지 대조
assert all(d in docs for d in rep_docs), 'rep_docs 는 실제 리뷰 본문이어야 합니다'
print("✅ 문제9 통과!")

## 10. 어디에도 안 묶인 문서 세기 — 노이즈(−1) 토픽
**배경**: BERTopic 안에서 문서를 묶는 **HDBSCAN** 은 **모든 문서를 억지로 어느 토픽엔가 배정하지 않습니다**. 어느 토픽에도 충분히 가깝지 않은 문서는 **−1(노이즈)** 로 남겨 둡니다. 노이즈가 몇 개인지 세어 보면, 이 토픽 모델이 리뷰의 몇 %를 실제 주제로 설명했는지 알 수 있습니다.

**요구사항**:
- 문제 7 의 `topics`(각 문서의 토픽 번호)를 사용하세요.
- 토픽 번호가 −1 인 문서가 몇 개인지 세어 변수 `n_noise` 에 **정수**로 담으세요.
- 전체 545개 중 노이즈 비율도 함께 출력해 보세요.

**예시**
```
n_noise            →  209 (전체 545개의 약 38%) — 컴퓨터에 따라 조금 흔들려도 150~280 이면 통과
n_noise 는 topic_info 의 Topic == -1 행의 Count 와 같아야 합니다
```
<details><summary>힌트</summary>

```text
접근방법:
- 각 문서의 토픽 번호 배열에서 -1 인 것만 세면 된다.
- 리스트는 불리언 마스크가 안 되므로 numpy 배열로 바꾼 뒤 비교한다.

세부구현:
1. topics 를 numpy 배열로 바꾼다
2. 그 배열이 -1 인지 비교해 만든 마스크의 합을 정수로 n_noise 에 담는다
3. n_noise 를 전체 문서 수로 나눠 비율을 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 지어낸 값이 아니라 실제 배정 결과인지 — topic_info 의 -1 행 Count 와 대조
_noise_row = topic_info.loc[topic_info['Topic'] == -1, 'Count']
assert len(_noise_row) == 1, '토픽 정보에 노이즈(-1) 행이 없습니다 — 표준 구성 그대로 학습했는지 확인하세요'
_info_noise = int(_noise_row.iloc[0])
assert n_noise == _info_noise, 'topics 에서 실제로 센 값이 아닙니다'
assert 150 <= n_noise <= 280, '노이즈 수가 예상 범위를 벗어났습니다 — 표준 구성 그대로 학습했는지 확인하세요'
print("✅ 문제10 통과! (노이즈:", n_noise, "개)")

## 11. 토픽별 문서 수 막대그래프
**배경**: 어떤 주제가 리뷰에서 **얼마나 큰 비중**을 차지하는지는 토픽별 문서 수를 막대그래프로 보면 한눈에 들어옵니다. 문제 7 의 `topic_info` 에서 노이즈(−1)를 뺀 토픽별 `Count` 를 막대로 그려 봅시다.

**요구사항**:
- 문제 7 의 `topic_info` 에서 **노이즈(−1) 행을 걸러낸** 표를 먼저 만드세요.
- `plt.figure(figsize=(8, 5))` 로 새 그림을 연 뒤, `plt.bar(...)` 로 x축은 토픽 번호(문자열), y축은 `Count` 인 막대그래프를 그리세요.
- 제목·축 이름을 달고 `plt.show()` 로 보여 주세요.

**예시**: 아래 완성 그래프와 같은 모양(토픽 번호별 문서 수 막대)이면 됩니다. (이 문제는 자가채점이 없습니다.)
<details><summary>힌트</summary>

```text
접근방법:
- 토픽 정보 표에서 노이즈(-1)를 걸러낸 뒤, 토픽 번호를 x, 문서 수(Count)를 y 로 막대그래프를 그린다.

세부구현:
1. topic_info 에서 Topic 이 -1 이 아닌 행만 고른다
2. plt.figure 로 새 그림을 연다
3. plt.bar 에 토픽 번호(문자열)와 Count 를 넘겨 막대를 그리고 제목·축 이름을 단 뒤 plt.show 로 보여 준다
```

</details>

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv1_q11_topicbar.png" width="600">

In [ ]:
# 여기에 코드를 작성하세요